# Neural Network: BP vs CLAPP vs CLAPP++DFB

Circle classification problem. Compare three training approaches on identical architectures:

1. **BP (Batch)** -- standard backprop with MSE loss, full-batch gradient descent
2. **CLAPP (Online)** -- contrastive layer-aware projection, triplets, online update, standard feedback
3. **CLAPP++DFB (Online)** -- same as CLAPP but with direct feedback from output layer

All trained on the circle-in-square problem: classify whether (x, y) is inside or outside a circle of radius r=0.75.

In [ ]:
import random
import math
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility
SEED = 42
random.seed(SEED)

In [ ]:
class NeuralNetworkParser:
    @staticmethod
    def parse(architecture: str, activation: str = "sigmoid") -> type:
        layer_sizes = [int(x) for x in architecture.split("-")]
        n_layers = len(layer_sizes)
        if n_layers < 2:
            raise ValueError("Architecture must have at least 2 layers")

        layer_names = [f"L{i}" for i in range(n_layers)]
        weight_keys = []
        for l in range(n_layers - 1):
            fn, tn = layer_names[l], layer_names[l + 1]
            for i in range(layer_sizes[l]):
                for j in range(layer_sizes[l + 1]):
                    weight_keys.append((fn, i, tn, j))

        cap = {
            "sizes": list(layer_sizes),
            "names": list(layer_names),
            "wkeys": list(weight_keys),
            "n": n_layers,
        }
        namespace = {"__module__": __name__}

        def __init__(self, lr=0.1, activation="sigmoid"):
            self._w = {k: random.uniform(-1.0, 1.0) for k in cap["wkeys"]}
            self._b = {ln: [0.0] * sz for ln, sz in zip(cap["names"], cap["sizes"])}
            self._lr = lr
            self._sizes = cap["sizes"]
            self._names = cap["names"]
            self._wkeys = cap["wkeys"]
            self._n = cap["n"]
            self._activation = activation
            self._input = None
            self._target = None
            self._last_act = None

        namespace["__init__"] = __init__

        def set_input(self, val):
            self._input = val

        namespace["set_input"] = set_input

        def set_target(self, tgt):
            if len(tgt) != self._sizes[-1]:
                raise ValueError("Target mismatch")
            self._target = tgt

        namespace["set_target"] = set_target

        @staticmethod
        def _sig(x):
            if x < -500:
                return 0.0
            if x > 500:
                return 1.0
            return 1.0 / (1.0 + math.exp(-x))

        @staticmethod
        def _relu(x):
            return max(0.0, x)

        @staticmethod
        def _sig_d(y):
            return y * (1.0 - y)

        @staticmethod
        def _relu_d(x):
            return 1.0 if x > 0.0 else 0.0

        def _activate(self, x):
            return self._relu(x) if self._activation == "relu" else self._sig(x)

        def _activate_d(self, x):
            if self._activation == "relu":
                return self._relu_d(x)
            return self._sig_d(self._activate(x))

        namespace.update(_sig=_sig, _relu=_relu, _sig_d=_sig_d, _relu_d=_relu_d)
        namespace["_activate"] = _activate
        namespace["_activate_d"] = _activate_d

        def _forward(self, x):
            if isinstance(x, (int, float)):
                x = [float(x)]
            else:
                x = list(x)
            act = {self._names[0]: x}
            for li in range(1, self._n):
                ln = self._names[li]
                pln = self._names[li - 1]
                prev = act[pln]
                raw = []
                for j in range(self._sizes[li]):
                    s = self._b[ln][j]
                    for i in range(self._sizes[li - 1]):
                        s += self._w[(pln, i, ln, j)] * prev[i]
                    raw.append(self._activate(s))
                act[ln] = raw
            return act

        namespace["_forward"] = _forward

        def compute(self) -> dict[str, float]:
            if self._input is None:
                raise ValueError("Input not set")
            self._last_act = self._forward(self._input)
            res = {}
            for li, ln in enumerate(self._names):
                for ni, v in enumerate(self._last_act[ln]):
                    res[f"a_{li}_{ni}"] = v
            return res

        namespace["compute"] = compute

        def _compute_gradients(self):
            if self._target is None:
                raise ValueError("Target not set")
            if self._last_act is None:
                raise ValueError("Run compute() first")
            act = self._last_act
            tgt = self._target
            out_ln = self._names[-1]
            out = act[out_ln]
            loss = sum((y - t) ** 2 for y, t in zip(out, tgt)) / len(tgt)
            deltas = {}
            for j, (y, t) in enumerate(zip(out, tgt)):
                deltas[(out_ln, j)] = 2 * (y - t) / len(tgt) * self._activate_d(y)
            for li in range(self._n - 2, 0, -1):
                ln = self._names[li]
                nln = self._names[li + 1]
                for i in range(self._sizes[li]):
                    ds = sum(self._w[(ln, i, nln, j)] * deltas[(nln, j)] for j in range(self._sizes[li + 1]))
                    deltas[(ln, i)] = self._activate_d(act[ln][i]) * ds
            wg, bg = {}, {}
            for pln, i, ln, j in self._wkeys:
                wg[(pln, i, ln, j)] = act[pln][i] * deltas[(ln, j)]
            for li in range(1, self._n):
                ln = self._names[li]
                for j in range(self._sizes[li]):
                    bg[(ln, j)] = deltas[(ln, j)]
            return wg, bg, loss

        namespace["_compute_gradients"] = _compute_gradients

        def train(self, dataset, epochs=100, verbose=10):
            n = len(dataset)
            if n == 0:
                raise ValueError("Empty dataset")
            hist = []
            acc_hist = []
            lr = self._lr
            for ep in range(epochs):
                wga = {k: 0.0 for k in self._wkeys}
                bga = {(ln, j): 0.0 for ln in self._names for j in range(self._sizes[self._names.index(ln)])}
                el = 0.0
                for x, y in dataset:
                    self._input = x
                    self._target = y
                    self._last_act = self._forward(x)
                    wg, bg, l = self._compute_gradients()
                    el += l
                    for k, g in wg.items():
                        wga[k] += g
                    for k, g in bg.items():
                        bga[k] += g
                for k in wga:
                    self._w[k] -= lr * (wga[k] / n)
                for k in bga:
                    self._b[k[0]][k[1]] -= lr * (bga[k] / n)
                avg = el / n
                hist.append(avg)
                acc_hist.append(_accuracy(self, dataset))
                if verbose > 0 and ep % verbose == 0:
                    print(f"Epoch {ep:4d}/{epochs}: loss = {avg:.6f}, acc = {acc_hist[-1]:.4f}")
            if verbose > 0 and epochs % verbose != 0:
                print(f"Epoch {epochs:4d}/{epochs}: loss = {hist[-1]:.6f}, acc = {acc_hist[-1]:.4f}")
            return hist, acc_hist

        namespace["train"] = train

        class_name = f"Network_{architecture.replace('-', 'x')}"
        Network = type(class_name, (object,), namespace)
        return Network


def _accuracy(net, data):
    c = 0
    for x, y in data:
        net._input = x
        out = net._forward(x)[net._names[-1]][0]
        pred = 1.0 if out >= 0.5 else 0.0
        if pred == y[0]:
            c += 1
    return c / len(data)

## Dataset: Circle Classification

Classify whether (x, y) is inside or outside a circle of radius r=0.75 centered at origin. Inputs normalized to [-1, 1].

In [ ]:
def in_circle(x: float, y: float, r: float = 0.75) -> float:
    return 0.0 if (x * x + y * y) <= (r * r) else 1.0

def make_dataset(grid=100, seed=SEED):
    random.seed(seed)
    dataset = []
    for i in range(grid):
        for j in range(grid):
            x = (i / (grid - 1)) * 2.0 - 1.0
            y = (j / (grid - 1)) * 2.0 - 1.0
            label = in_circle(x, y)
            dataset.append(([x, y], [label]))
    random.shuffle(dataset)
    return dataset

def split_train_test(dataset, train_frac=0.8, seed=SEED):
    random.seed(seed)
    random.shuffle(dataset)
    n = int(len(dataset) * train_frac)
    return dataset[:n], dataset[n:]

full_dataset = make_dataset()
train_data, test_data = split_train_test(full_dataset)
print(f"Train: {len(train_data)}, Test: {len(test_data)}")

## CLAPP & CLAPP++DFB Training

Attach contrastive training methods to the network class.

In [ ]:
def attach_clapp_methods(NetworkClass):
    if hasattr(NetworkClass, "train_clapp"):
        return NetworkClass

    def _softplus(x: float) -> float:
        if x > 30.0:
            return x
        if x < -30.0:
            return math.exp(x)
        return math.log1p(math.exp(x))

    def _logistic(x: float) -> float:
        if x >= 0.0:
            e = math.exp(-x)
            return 1.0 / (1.0 + e)
        e = math.exp(x)
        return e / (1.0 + e)

    def _clapp_step(self, x_anchor, x_context, x_negative, lambda_b, beta_b, direct_feedback):
        act_p = self._forward(x_anchor)
        act_c = self._forward(x_context)
        act_n = self._forward(x_negative)

        total_loss = 0.0
        wg = {k: 0.0 for k in self._wkeys}
        bg = {(ln, j): 0.0 for ln in self._names for j in range(self._sizes[self._names.index(ln)])}
        bg_b = {}

        for li in range(1, self._n):
            ln = self._names[li]
            pln = self._names[li - 1]
            z_p = act_p[ln]
            z_n = act_n[ln]
            c_vec = act_c[self._names[-1]] if direct_feedback else act_c[ln]
            B = self._clapp_B[ln]

            u = [sum(B[j][k] * c_vec[k] for k in range(len(c_vec))) for j in range(len(z_p))]
            s_p = sum(z_p[j] * u[j] for j in range(len(z_p)))
            s_n = sum(z_n[j] * u[j] for j in range(len(z_n)))

            loss_l = _softplus(-s_p) + _softplus(s_n)
            reg = sum(v * v for row in B for v in row)
            loss_l += lambda_b * reg
            total_loss += loss_l

            g_p = -_logistic(-s_p)
            g_n = _logistic(s_n)

            for j in range(len(B)):
                for k in range(len(c_vec)):
                    bg_b[(ln, j, k)] = (
                        g_p * z_p[j] * c_vec[k] + g_n * z_n[j] * c_vec[k] + 2.0 * lambda_b * B[j][k]
                    )

            dp = [self._activate_d(v) for v in z_p]
            dn = [self._activate_d(v) for v in z_n]
            zp_prev = act_p[pln]
            zn_prev = act_n[pln]

            for i in range(self._sizes[li - 1]):
                for j in range(self._sizes[li]):
                    wg[(pln, i, ln, j)] += (
                        g_p * u[j] * dp[j] * zp_prev[i] + g_n * u[j] * dn[j] * zn_prev[i]
                    )

            for j in range(self._sizes[li]):
                bg[(ln, j)] += g_p * u[j] * dp[j] + g_n * u[j] * dn[j]

        # Online update (immediate, per sample)
        for k in wg:
            self._w[k] -= self._lr * wg[k]
        for k in bg:
            self._b[k[0]][k[1]] -= self._lr * bg[k]
        for (ln, j, k), g in bg_b.items():
            self._clapp_B[ln][j][k] -= beta_b * g

        return total_loss

    def train_clapp(self, triplets, epochs=100, verbose=10, lambda_b=1e-3,
                    beta_b=None, direct_feedback=False, shuffle=True):
        n = len(triplets)
        if n == 0:
            raise ValueError("Triplet list is empty.")
        if beta_b is None:
            beta_b = self._lr * 10.0

        if not hasattr(self, "_clapp_B"):
            self._clapp_B = {}
        for li in range(1, self._n):
            ln = self._names[li]
            ctx_dim = self._sizes[-1] if direct_feedback else self._sizes[li]
            self._clapp_B[ln] = [
                [random.uniform(-0.1, 0.1) for _ in range(ctx_dim)]
                for _ in range(self._sizes[li])
            ]

        hist = []
        acc_hist = []
        for ep in range(epochs):
            if shuffle:
                random.shuffle(triplets)
            el = 0.0
            for xa, xc, xn in triplets:
                l = self._clapp_step(xa, xc, xn, lambda_b, beta_b, direct_feedback)
                el += l
            avg = el / n
            hist.append(avg)
            acc_hist.append(_accuracy(self, train_data))
            if verbose > 0 and ep % verbose == 0:
                print(f"Epoch {ep:4d}/{epochs}: loss = {avg:.6f}, acc = {acc_hist[-1]:.4f}")
        if verbose > 0 and epochs % verbose != 0:
            print(f"Epoch {epochs:4d}/{epochs}: loss = {hist[-1]:.6f}, acc = {acc_hist[-1]:.4f}")
        return hist, acc_hist

    def get_clapp_projections(self):
        if not hasattr(self, "_clapp_B"):
            return {}
        return {k: [row[:] for row in v] for k, v in self._clapp_B.items()}

    NetworkClass._softplus = staticmethod(_softplus)
    NetworkClass._logistic = staticmethod(_logistic)
    NetworkClass._clapp_step = _clapp_step
    NetworkClass.train_clapp = train_clapp
    NetworkClass.get_clapp_projections = get_clapp_projections
    return NetworkClass


def build_clapp_triplets_from_labeled(dataset):
    by_label = {}
    for x, y in dataset:
        lbl = tuple(y)
        by_label.setdefault(lbl, []).append(list(x))
    labels = list(by_label.keys())
    triplets = []
    for x, y in dataset:
        lbl = tuple(y)
        x_ctx = random.choice(by_label[lbl])
        neg_lbl = random.choice([k for k in labels if k != lbl]) if len(labels) > 1 else lbl
        x_neg = random.choice(by_label[neg_lbl])
        triplets.append((list(x), list(x_ctx), list(x_neg)))
    return triplets

## Train All Three Methods

Identical architecture `2-10-10-1`, ReLU activation, same learning rate. BP uses batch MSE, CLAPP variants use online contrastive loss.

In [ ]:
ACT = "relu"
ARCH = "2-10-10-1"
N_EPOCHS = 200
LR = 0.05
VERBOSE = 50

triplets = build_clapp_triplets_from_labeled(train_data)

# --- BP (Batch) ---
print("=== BP (Batch) ===")
NetBP = NeuralNetworkParser.parse(ARCH, activation=ACT)
nn_bp = NetBP(lr=LR, activation=ACT)
hist_bp, acc_bp = nn_bp.train(train_data, epochs=N_EPOCHS, verbose=VERBOSE)

# --- CLAPP (Online, standard feedback) ---
print("\n=== CLAPP (Online) ===")
NetCL = attach_clapp_methods(NeuralNetworkParser.parse(ARCH, activation=ACT))
nn_cl = NetCL(lr=LR, activation=ACT)
hist_cl, acc_cl = nn_cl.train_clapp(triplets, epochs=N_EPOCHS, verbose=VERBOSE,
                                     direct_feedback=False, beta_b=LR * 10)

# --- CLAPP++DFB (Online, direct feedback) ---
print("\n=== CLAPP++DFB (Online) ===")
NetDF = attach_clapp_methods(NeuralNetworkParser.parse(ARCH, activation=ACT))
nn_df = NetDF(lr=LR, activation=ACT)
hist_df, acc_df = nn_df.train_clapp(triplets, epochs=N_EPOCHS, verbose=VERBOSE,
                                     direct_feedback=True, beta_b=LR * 10)

## Results

In [ ]:
def evaluate(net, data):
    c = 0
    for x, y in data:
        net.set_input(x)
        out = net.compute()["a_3_0"]
        pred = 1.0 if out >= 0.5 else 0.0
        if pred == y[0]:
            c += 1
    return c / len(data)

print(f"{'Method':<20} {'Train Acc':>10} {'Test Acc':>10}")
print(f"{'=' * 42}")
print(f"{'BP (Batch)':<20} {evaluate(nn_bp, train_data):>10.4f} {evaluate(nn_bp, test_data):>10.4f}")
print(f"{'CLAPP (Online)':<20} {evaluate(nn_cl, train_data):>10.4f} {evaluate(nn_cl, test_data):>10.4f}")
print(f"{'CLAPP++DFB':<20} {evaluate(nn_df, train_data):>10.4f} {evaluate(nn_df, test_data):>10.4f}")

## Accuracy Over Epochs

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(acc_bp, label="BP (Batch)", linewidth=2)
plt.plot(acc_cl, label="CLAPP (Online)", linestyle="--", linewidth=2)
plt.plot(acc_df, label="CLAPP++DFB (Online)", linestyle="-.", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Train Accuracy")
plt.title(f"Accuracy Over Epochs (arch={ARCH}, act={ACT}, lr={LR})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.0, 1.05)
plt.tight_layout()
plt.savefig("nn_accuracy.png", dpi=150)
plt.show()

## Decision Boundaries

In [ ]:
def plot_decision_boundary(net, title, ax, grid_points=100):
    xs = np.linspace(-1, 1, grid_points)
    ys = np.linspace(-1, 1, grid_points)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(grid_points):
        for j in range(grid_points):
            net.set_input([X[i, j], Y[i, j]])
            Z[i, j] = net.compute()["a_3_0"]
    ax.contourf(X, Y, Z, levels=20, cmap="RdBu_r", alpha=0.8)
    ax.contour(X, Y, Z, levels=[0.5], colors="black", linewidths=1.5)
    r = 0.75
    circle = plt.Circle((0, 0), r, color="orange", fill=False, linewidth=2, linestyle="--", label="True boundary (r=0.75)")
    ax.add_patch(circle)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect("equal")
    

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

plot_decision_boundary(nn_bp, "BP (Batch)", axes[0])
plot_decision_boundary(nn_cl, "CLAPP (Online)", axes[1])
plot_decision_boundary(nn_df, "CLAPP++DFB", axes[2])

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Decision Boundaries", fontsize=14)
plt.tight_layout()
plt.savefig("nn_decision_boundaries.png", dpi=150)
plt.show()

## 1D Slices

In [ ]:
def plot_1d_slices(net, title):
    x = np.linspace(-1, 1, 200)
    y0_vals = []
    y1_vals = []
    for xi in x:
        net.set_input([0, xi])
        y0_vals.append(net.compute()["a_3_0"])
        net.set_input([xi, 0])
        y1_vals.append(net.compute()["a_3_0"])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(x, y0_vals)
    ax1.set_title(f"{title}: output for [0, y]")
    ax1.set_xlabel("y")
    ax1.set_ylabel("output")
    ax1.grid(True, alpha=0.3)
    ax2.plot(x, y1_vals)
    ax2.set_title(f"{title}: output for [x, 0]")
    ax2.set_xlabel("x")
    ax2.set_ylabel("output")
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_1d_slices(nn_bp, "BP")
plot_1d_slices(nn_cl, "CLAPP")
plot_1d_slices(nn_df, "CLAPP++DFB")
